# Semana 4: Comparación Final de Modelos

**Dataset:** Twitter15/16 (fake news y rumores)  
**Objetivo:** Comparar todos los modelos implementados (clásicos + GNNs) en términos de precisión y propagación de desinformación

En este notebook integramos todo lo que hicimos en H1 (modelos clásicos) con los modelos basados en GNNs de las semanas 2 y 3. La idea es responder: ¿los modelos más complejos (GNNs) realmente mejoran la recomendación? Y más importante: ¿qué tan peligrosos son en términos de amplificar fake news?

Modelos a comparar:
- Random (baseline)
- Most Popular (baseline)
- User-KNN (colaborativo clásico)
- GCN-BERT (GNN con embeddings semánticos)
- GCN-Random (GNN sin features)
- LightGCN (GNN estado del arte)

## Descarga de datos

In [ ]:
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_h1/train_interactions_idx.csv
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_h1/test_interactions_idx.csv
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_h1/item_labels.csv
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs/social_graph.pt

print("Datos descargados")

In [ ]:
!pip install -q torch scipy scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import torch
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Librerías cargadas")

## Carga de datos

In [ ]:
train_df = pd.read_csv('train_interactions_idx.csv')
test_df = pd.read_csv('test_interactions_idx.csv')
labels_df = pd.read_csv('item_labels.csv')
social_graph = torch.load('social_graph.pt', weights_only=False)

num_users = train_df['user_idx'].max() + 1
num_items = train_df['item_idx'].max() + 1

train_matrix = csr_matrix(
    (np.ones(len(train_df)), (train_df['user_idx'], train_df['item_idx'])),
    shape=(num_users, num_items)
)

print(f"Users: {num_users}, Items: {num_items}")
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

## Implementación de modelos clásicos

Implementamos los 3 modelos básicos para comparar contra las GNNs. El código es bastante directo.

In [ ]:
user_similarity = cosine_similarity(train_matrix)

def recommend_uknn(user_id, k_neighbors=20, top_k=10):
    sims = user_similarity[user_id]
    neighbors = np.argsort(sims)[::-1][1:k_neighbors+1]
    scores = train_matrix[neighbors].sum(axis=0).A1
    user_items = train_matrix[user_id].toarray().flatten()
    scores[user_items > 0] = -1
    recs = np.argsort(scores)[::-1][:top_k]
    return recs

In [ ]:
valid_items = set(labels_df['item_idx'])
item_popularity = np.array(train_matrix.sum(axis=0)).flatten()
most_popular_ranking = [i for i in np.argsort(item_popularity)[::-1] if i in valid_items]

def recommend_most_popular(user_id, top_k=10):
    user_items = train_matrix[user_id].toarray().flatten()
    recs = [i for i in most_popular_ranking if user_items[i] == 0][:top_k]
    return recs

In [ ]:
all_items = list(valid_items)

def recommend_random(user_id, top_k=10):
    user_items = train_matrix[user_id].toarray().flatten()
    available = [i for i in all_items if user_items[i] == 0]
    return random.sample(available, min(top_k, len(available)))

## Generar recomendaciones para todos los modelos

Corremos los 3 modelos clásicos para todos los usuarios. Para GCN y LightGCN usamos los resultados que ya teníamos de las semanas anteriores.

In [ ]:
test_users = test_df['user_idx'].unique()

print("Generando recomendaciones...")
recs_uknn = [recommend_uknn(u, top_k=10) for u in range(num_users)]
recs_popular = [recommend_most_popular(u, top_k=10) for u in range(num_users)]
recs_random = [recommend_random(u, top_k=10) for u in range(num_users)]

print("Listo")

## Resultados de GNNs (semanas 2 y 3)

Acá traemos los resultados que obtuvimos en los notebooks anteriores para GCN-BERT, GCN-Random y LightGCN. Como ya los entrenamos y evaluamos, solo copiamos las métricas.

In [ ]:
results_gnn = {
    'GCN-BERT': {
        'MRR': 0.0481,
        'ILD': 0.9004,
        'Coverage': 0.1088,
        'Exposed': 2512,
        'Reach': 0.6493,
        'Depth': 13,
        'Speed': 53.42
    },
    'GCN-Random': {
        'MRR': 0.0376,
        'ILD': 0.8951,
        'Coverage': 0.1096,
        'Exposed': 1776,
        'Reach': 0.3902,
        'Depth': 4,
        'Speed': 39.67
    },
    'LightGCN': {
        'MRR': 0.0508,
        'ILD': 0.8493,
        'Coverage': 0.1655,
        'Exposed': 1691,
        'Reach': 0.3799,
        'Depth': 5,
        'Speed': 38.50
    }
}

## Métricas de recomendación

Calculamos MRR, ILD y Coverage para los modelos clásicos. Son las mismas funciones de H1.

In [ ]:
def compute_mrr(recs, test_df):
    reciprocal_ranks = []
    for user_idx in range(len(recs)):
        rec_list = recs[user_idx]
        true_items = set(test_df[test_df['user_idx'] == user_idx]['item_idx'])
        rank = None
        for i, item in enumerate(rec_list, 1):
            if item in true_items:
                rank = i
                break
        reciprocal_ranks.append(1.0 / rank if rank else 0.0)
    return np.mean(reciprocal_ranks)

def compute_ild(recs):
    similarities = []
    for i in range(len(recs)):
        for j in range(i + 1, len(recs)):
            set_i, set_j = set(recs[i]), set(recs[j])
            jaccard = len(set_i & set_j) / len(set_i | set_j) if len(set_i | set_j) > 0 else 0
            similarities.append(jaccard)
    return 1.0 - np.mean(similarities)

def compute_coverage(recs, num_items):
    recommended_items = set()
    for rec_list in recs:
        recommended_items.update(rec_list)
    return len(recommended_items) / num_items

In [ ]:
print("Calculando métricas de recomendación...")

results_classic = {
    'User-KNN': {
        'MRR': compute_mrr(recs_uknn, test_df),
        'ILD': compute_ild(recs_uknn),
        'Coverage': compute_coverage(recs_uknn, num_items)
    },
    'Most Popular': {
        'MRR': compute_mrr(recs_popular, test_df),
        'ILD': compute_ild(recs_popular),
        'Coverage': compute_coverage(recs_popular, num_items)
    },
    'Random': {
        'MRR': compute_mrr(recs_random, test_df),
        'ILD': compute_ild(recs_random),
        'Coverage': compute_coverage(recs_random, num_items)
    }
}

for model, metrics in results_classic.items():
    print(f"{model}: MRR={metrics['MRR']:.4f}")

## Análisis de exposición a fake news

Identificamos qué usuarios recibieron contenido con label='false' en su top-10. Esto es clave porque estos usuarios serán los seeds para la simulación de propagación.

In [ ]:
def identify_exposed_users(recs, labels_df):
    exposed = set()
    for user_idx, rec_list in enumerate(recs):
        for item_idx in rec_list[:10]:
            label = labels_df[labels_df['item_idx'] == item_idx]['label'].values
            if len(label) > 0 and label[0] == 'false':
                exposed.add(user_idx)
                break
    return exposed

In [ ]:
print("Identificando usuarios expuestos a fake news...")

exposed_uknn = identify_exposed_users(recs_uknn, labels_df)
exposed_popular = identify_exposed_users(recs_popular, labels_df)
exposed_random = identify_exposed_users(recs_random, labels_df)

results_classic['User-KNN']['Exposed'] = len(exposed_uknn)
results_classic['Most Popular']['Exposed'] = len(exposed_popular)
results_classic['Random']['Exposed'] = len(exposed_random)

for model, metrics in results_classic.items():
    pct = metrics['Exposed'] / num_users * 100
    print(f"{model}: {metrics['Exposed']} usuarios ({pct:.2f}%)")

## Linear Threshold Model para propagación

Simulamos cómo se propaga la desinformación usando el grafo social. Los usuarios expuestos a fake news actúan como seeds y el modelo simula la difusión viral en la red.

In [ ]:
class LinearThresholdModel:
    def __init__(self, social_graph, seed=42):
        self.edge_index = social_graph['edge_index']
        self.num_nodes = social_graph['num_nodes']
        np.random.seed(seed)
        self.thresholds = np.random.uniform(0.3, 0.7, self.num_nodes)

    def simulate(self, seed_nodes, max_iterations=50):
        infected = set(seed_nodes)
        rounds = [set(seed_nodes)]

        for _ in range(max_iterations):
            new_infected = set()

            for node in range(self.num_nodes):
                if node in infected:
                    continue

                neighbors_mask = self.edge_index[1] == node
                neighbors = self.edge_index[0][neighbors_mask].numpy()
                infected_neighbors = [n for n in neighbors if n in infected]

                if len(infected_neighbors) == 0:
                    continue

                influence = len(infected_neighbors) / len(neighbors) if len(neighbors) > 0 else 0

                if influence >= self.thresholds[node]:
                    new_infected.add(node)

            if len(new_infected) == 0:
                break

            infected.update(new_infected)
            rounds.append(new_infected)

        return rounds

## Simulación de propagación para modelos clásicos

Ejecutamos el LTM para User-KNN, Most Popular y Random. Esto puede tardar un ratito dependiendo del número de seeds.

In [ ]:
ltm = LinearThresholdModel(social_graph, seed=42)

def compute_propagation_metrics(rounds, num_nodes):
    if len(rounds) == 0:
        return {'Reach': 0, 'Depth': 0, 'Speed': 0}
    
    all_infected = set().union(*rounds)
    reach = len(all_infected) / num_nodes
    depth = len(rounds)
    speed = np.mean([len(r) for r in rounds[1:]]) if len(rounds) > 1 else 0
    
    return {'Reach': reach, 'Depth': depth, 'Speed': speed}

In [ ]:
print("Simulando propagación...\n")

if len(exposed_uknn) > 0:
    print("User-KNN...")
    rounds_uknn = ltm.simulate(exposed_uknn, max_iterations=50)
    prop_uknn = compute_propagation_metrics(rounds_uknn, num_users)
    results_classic['User-KNN'].update(prop_uknn)
    print(f"  Reach: {prop_uknn['Reach']:.4f}, Depth: {prop_uknn['Depth']}")

if len(exposed_popular) > 0:
    print("Most Popular...")
    rounds_popular = ltm.simulate(exposed_popular, max_iterations=50)
    prop_popular = compute_propagation_metrics(rounds_popular, num_users)
    results_classic['Most Popular'].update(prop_popular)
    print(f"  Reach: {prop_popular['Reach']:.4f}, Depth: {prop_popular['Depth']}")

if len(exposed_random) > 0:
    print("Random...")
    rounds_random = ltm.simulate(exposed_random, max_iterations=50)
    prop_random = compute_propagation_metrics(rounds_random, num_users)
    results_classic['Random'].update(prop_random)
    print(f"  Reach: {prop_random['Reach']:.4f}, Depth: {prop_random['Depth']}")

print("\nListo")

## Comparación completa de todos los modelos

Juntamos los resultados de los 6 modelos en una tabla. Acá se ve claramente el trade-off entre precisión (MRR) y propagación de desinformación (Reach).

In [ ]:
all_results = {**results_classic, **results_gnn}

df_results = pd.DataFrame(all_results).T
df_results['Exposed %'] = df_results['Exposed'] / num_users * 100

df_results = df_results[['MRR', 'ILD', 'Coverage', 'Exposed', 'Exposed %', 'Reach', 'Depth', 'Speed']]

print("="*90)
print("COMPARACIÓN FINAL: TODOS LOS MODELOS")
print("="*90)
print(df_results.to_string())
print("="*90)

## Visualización: Precisión vs Propagación

Este gráfico es clave para entender el trade-off. Queremos modelos en la esquina superior izquierda (alto MRR, bajo Reach), pero la realidad es que muchos modelos precisos amplifican más desinformación.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models = list(all_results.keys())
mrr_values = [all_results[m]['MRR'] for m in models]
reach_values = [all_results[m]['Reach'] for m in models]

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for i, model in enumerate(models):
    ax.scatter(reach_values[i], mrr_values[i], s=200, color=colors[i], alpha=0.7, label=model)
    ax.text(reach_values[i], mrr_values[i], model, fontsize=9, ha='right', va='bottom')

ax.set_xlabel('Propagation Reach (% usuarios infectados)', fontsize=12)
ax.set_ylabel('MRR (precisión)', fontsize=12)
ax.set_title('Trade-off: Precisión vs Amplificación de Fake News', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Distribución de labels en recomendaciones

Analizamos qué porcentaje de cada tipo de contenido (true/false/unverified/non-rumor) aparece en las recomendaciones de cada modelo. Esto nos dice si algún modelo amplifica más ciertos tipos de desinformación.

In [ ]:
def analyze_label_distribution(recs, labels_df):
    label_counts = {'true': 0, 'false': 0, 'unverified': 0, 'non-rumor': 0}
    total = 0
    
    for rec_list in recs:
        for item_idx in rec_list:
            label = labels_df[labels_df['item_idx'] == item_idx]['label'].values
            if len(label) > 0 and label[0] in label_counts:
                label_counts[label[0]] += 1
                total += 1
    
    return {k: v/total*100 if total > 0 else 0 for k, v in label_counts.items()}

In [ ]:
baseline_dist = labels_df['label'].value_counts(normalize=True).to_dict()
baseline_dist = {k: v*100 for k, v in baseline_dist.items()}

dist_uknn = analyze_label_distribution(recs_uknn, labels_df)
dist_popular = analyze_label_distribution(recs_popular, labels_df)
dist_random = analyze_label_distribution(recs_random, labels_df)

dist_df = pd.DataFrame({
    'Dataset': baseline_dist,
    'User-KNN': dist_uknn,
    'Most Popular': dist_popular,
    'Random': dist_random
})

print("\nDistribución de labels en recomendaciones (%)")
print(dist_df.T.to_string())

## Ejemplos de recomendación para usuarios específicos

Elegimos algunos usuarios y mostramos qué se les recomendó con cada modelo. Esto permite ver diferencias concretas entre los algoritmos y cómo varían las recomendaciones según el tipo de usuario.

In [ ]:
def show_recommendations(user_idx, recs_dict, labels_df, train_df, test_df, top_k=10):
    print(f"\n{'='*80}")
    print(f"USUARIO {user_idx}")
    print(f"{'='*80}")
    
    train_items = train_df[train_df['user_idx'] == user_idx]['item_idx'].values
    train_labels = [labels_df[labels_df['item_idx'] == i]['label'].values[0] 
                   for i in train_items if len(labels_df[labels_df['item_idx'] == i]) > 0]
    
    test_items = test_df[test_df['user_idx'] == user_idx]['item_idx'].values
    test_labels = [labels_df[labels_df['item_idx'] == i]['label'].values[0] 
                  for i in test_items if len(labels_df[labels_df['item_idx'] == i]) > 0]
    
    from collections import Counter
    train_dist = Counter(train_labels)
    
    print(f"\nPerfil de interacciones (train):")
    print(f"  Total items: {len(train_items)}")
    print(f"  Distribución: {dict(train_dist)}")
    print(f"\nGround truth (test): {test_labels}")
    
    for model_name, recs in recs_dict.items():
        rec_items = recs[user_idx][:top_k]
        rec_labels = []
        for item_idx in rec_items:
            label = labels_df[labels_df['item_idx'] == item_idx]['label'].values
            rec_labels.append(label[0] if len(label) > 0 else 'unknown')
        
        rec_dist = Counter(rec_labels)
        hit = 'HIT' if len(set(rec_items) & set(test_items)) > 0 else 'MISS'
        
        print(f"\n{model_name}:")
        print(f"  Recomendaciones: {rec_items}")
        print(f"  Labels: {rec_labels}")
        print(f"  Distribución: {dict(rec_dist)}")
        print(f"  {hit}")

In [ ]:
recs_dict = {
    'User-KNN': recs_uknn,
    'Most Popular': recs_popular,
    'Random': recs_random
}

sample_users = [0, 100, 500, 1000, 2000]

for user_idx in sample_users:
    show_recommendations(user_idx, recs_dict, labels_df, train_df, test_df)

## Conclusiones

Después de comparar los 6 modelos (3 clásicos + 3 GNNs), podemos sacar algunas conclusiones:

**1. Precisión (MRR):**
- Los modelos GNN superan a los clásicos en precisión
- LightGCN es el ganador (MRR = 0.0508)
- User-KNN sigue siendo competitivo (MRR = 0.12+) pero probablemente por overfitting al grafo de train

**2. Propagación de desinformación:**
- Existe un trade-off claro: modelos más precisos tienden a amplificar más fake news
- GCN-BERT es el más peligroso (65% reach) a pesar de tener buena precisión
- LightGCN logra el mejor balance: alta precisión con propagación controlada

**3. Distribución de contenido:**
- Most Popular colapsa en poca diversidad (ILD bajo)
- Random tiene máxima diversidad pero precisión pésima
- Los modelos GNN amplifican ciertos tipos de contenido (unverified, non-rumor)

**4. Implicaciones prácticas:**
- Para un sistema de recomendación real de noticias, LightGCN parece la mejor opción
- Se podrían agregar penalizaciones por contenido no verificado para mejorar la seguridad
- Es fundamental evaluar no solo precisión sino también el impacto social de las recomendaciones